In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re

sns.set(style="whitegrid", context="talk")

# =============================
# CONFIG
# =============================
DATA_DIR = Path("/home/alecacciatore/rolex/deep-nir/src/deep_nir/VRAI/outputs/multi_dataset_training")
FAMILIES = ["DM", "Starch", "Protein", "ADF", "NDF", "Ash", "Crude Fat", "Crude Fib."]
METRICS = ["SEP", "SEPC", "Bias", "R2", "Slope"]
OTHER_METRICS = ["N", "Min_True", "Max_True", "Avg_True", "Std_True", "Min_Pred", "Max_Pred", "Avg_Pred", "Std_Pred"]


# =============================
# PARSE FILENAME
# =============================
def parse_filename(fname):
    """
    Returns: family, regressor, mode
    """
    name = fname.stem
    mode = "each8" if "each8" in name else "global"

    for fam in FAMILIES:
        if fam in name:
            family = fam
            break

    regressor = name.replace(family, "").replace("_SVM", "").replace("_each8", "").strip("_")
    return family, regressor, mode


# =============================
# LOAD ALL FILES
# =============================
def load_results():
    rows = []

    for file in DATA_DIR.glob("*.csv"):
        family, regressor, mode = parse_filename(file)
        df = pd.read_csv(file)

        for metric in METRICS:
            train_col = f"{metric}_{family}"
            val_col = f"val_{metric}_{family}"

            if train_col in df.columns:
                rows.append({
                    "Family": family,
                    "Regressor": regressor,
                    "Mode": mode,
                    "Metric": metric,
                    "Dataset": "Train",
                    "Value": df[train_col].iloc[0]
                })

            if val_col in df.columns:
                rows.append({
                    "Family": family,
                    "Regressor": regressor,
                    "Mode": mode,
                    "Metric": metric,
                    "Dataset": "Validation",
                    "Value": df[val_col].iloc[0]
                })

    return pd.DataFrame(rows)


df = load_results()

DM ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
DM ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
Starch ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
Protein ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
ADF ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
NDF ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
Ash ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
Crude Fat ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
DM ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
Starch ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
DM ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
Starch ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']
Protein ['DM', 'Starch', 

UnboundLocalError: local variable 'family' referenced before assignment

# 1. Scatter R² vs SEP (Validation)

In [ ]:
plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df.query("Metric in ['R2', 'SEP'] and Dataset=='Validation'")
      .pivot_table(index=["Family","Regressor","Mode"],
                   columns="Metric",
                   values="Value")
      .reset_index(),
    x="SEP",
    y="R2",
    hue="Mode",
    style="Family",
    s=120
)

plt.title("Validation performance: R² vs SEP")
plt.tight_layout()
plt.show()

# 2. Bar plot accoppiato SEP / SEPC (per famiglia)

In [ ]:
g = sns.catplot(
    data=df.query("Metric in ['SEP','SEPC'] and Dataset=='Validation'"),
    x="Metric",
    y="Value",
    hue="Mode",
    col="Family",
    kind="bar",
    col_wrap=4,
    sharey=False,
    height=4
)

g.fig.suptitle("Validation SEP / SEPC per famiglia", y=1.05)
plt.show()


# 3. Bias & Slope → stabilità del modello

In [ ]:
g = sns.catplot(
    data=df.query("Metric in ['Bias','Slope'] and Dataset=='Validation'"),
    x="Metric",
    y="Value",
    hue="Mode",
    col="Family",
    kind="point",
    col_wrap=4,
    height=4
)

g.fig.suptitle("Bias & Slope (Validation)", y=1.05)
plt.show()


# 4. Train vs Validation (overfitting check)

In [ ]:
g = sns.catplot(
    data=df.query("Metric=='R2'"),
    x="Dataset",
    y="Value",
    hue="Mode",
    col="Family",
    kind="bar",
    col_wrap=4,
    height=4
)

g.fig.suptitle("Train vs Validation R²", y=1.05)
plt.show()
